In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

# 少样本提示：先给AI两个示例，再让它生成新菜谱
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是一个菜谱助手。请严格按照示例格式输出。"},
        {"role": "user", "content": """
示例1：
菜名：番茄炒蛋
食材：番茄2个，鸡蛋3个，盐，糖
步骤：1. 番茄切块，鸡蛋打散。2. 炒鸡蛋盛出。3. 炒番茄，加入鸡蛋。4. 调味出锅。

示例2：
菜名：青椒肉丝
食材：青椒2个，瘦肉200g，生抽，淀粉
步骤：1. 肉切丝加生抽淀粉腌制。2. 青椒切丝。3. 先炒肉丝变色盛出。4. 炒青椒，加入肉丝。

现在请用同样的格式，生成一道“红烧肉”的菜谱。
"""}
    ],
    temperature=0.5,  # 中等温度，既有稳定性又有些创意
    max_tokens=500
)

print(response.choices[0].message.content)
print(response.usage.total_tokens)

菜名：红烧肉  
食材：五花肉500g，生姜3片，葱1根，料酒，生抽，老抽，冰糖，盐  
步骤：1. 五花肉切块，冷水下锅加料酒焯水。2. 锅中放油炒冰糖至融化。3. 加入肉块翻炒上色。4. 加生抽、老抽、姜、葱和水炖煮。5. 小火炖至软烂，收汁调味。
262


In [4]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是一个数学老师，请分步推理。"},
        {"role": "user", "content": """
严格按照示例格式进行输出
问题：小明买了3个苹果，每个苹果2元，又买了5个橘子，每个橘子3元。他付了50元，应找回多少钱？

让我们一步一步思考：
1. 先计算苹果的总价：3个 × 2元 = 6元
2. 再计算橘子的总价：5个 × 3元 = 15元
3. 总花费：6 + 15 = 21元
4. 应找回：50 - 21 = 29元

答案：29元。

现在请用同样的分步推理方式，解决这个问题：
问题：小红买了4支铅笔，每支1.5元，又买了2个笔记本，每个4元。她付了20元，应找回多少钱？
"""}
    ],
    temperature=0.3,  # 数学题需要低温度保证准确
    max_tokens=300
)

print(response.choices[0].message.content)

让我们一步一步思考：  
1. 先计算铅笔的总价：4支 × 1.5元 = 6元  
2. 再计算笔记本的总价：2个 × 4元 = 8元  
3. 总花费：6 + 8 = 14元  
4. 应找回：20 - 14 = 6元  

答案：6元。


In [5]:
# 你可以自由发挥，例如让AI生成一个学习计划，并说明理由
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是一个学习规划师。请用示例格式输出。"},
        {"role": "user", "content": """
请一步一步的思考然后严格按照示例格式输出，体现思考过程
示例学习计划：
分布思考：
日期：周一
时间：19:00-20:00
内容：复习Python基础
理由：巩固基础，为后续项目做准备。

现在请为今天（周四）晚上制定一个学习计划，包含时间段、内容和理由。
"""}
    ],
    temperature=0.5,
    max_tokens=300
)

print(response.choices[0].message.content)

分布思考：  
1. 今天是周四，需要制定晚上的学习计划。  
2. 晚上时间通常适合安排1-2小时的学习，假设从19:00开始。  
3. 最近可能在学数据分析，可以安排复习相关技能。  
4. 内容选择复习Pandas数据处理，因为它是数据分析的核心工具。  
5. 理由：巩固数据处理能力，为明天的实战练习打基础。  

日期：周四  
时间：19:00-20:30  
内容：复习Pandas数据处理（如数据清洗、转换）  
理由：巩固核心技能，提升实战效率，为明天项目做准备。


In [13]:
!pip install langchain langchain-community langchain-openai

Defaulting to user installation because normal site-packages is not writeable


In [1]:
from langchain_community.document_loaders import TextLoader

# 加载你刚才保存的txt文件（以notice.txt为例）
loader = TextLoader("./data/notice.txt", encoding="utf-8")
documents = loader.load()

print(f"加载了 {len(documents)} 个文档")
print("文档内容预览：")
print(documents[0].page_content[:200])  # 打印前200个字符

加载了 1 个文档
文档内容预览：
浙江大学教务处通知汇编
═══════════════════════════════════════════════════════

【通知一】关于2025-2026学年秋冬学期本科课程第四轮选课的通知
发布时间：2025年10月30日
内容：2025-2026学年秋冬学期本科课程第四轮选课安排

【通知二】2025级本科生主修专业确认工作的通知
发布时间：2025年10月30日
内容：202


In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 创建分割器，设置块大小和重叠
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,        # 每个块的最大字符数（可根据文档调整）
    chunk_overlap=20,      # 块之间的重叠字符数，防止语义断裂
    length_function=len,   # 计算长度的函数，默认len
    separators=["\n\n", "\n", "。", "！", "？", "，", " ", ""]  # 优先按段落、句子分割
)

# 对文档进行切分
chunks = text_splitter.split_documents(documents)

print(f"原始文档被切分成 {len(chunks)} 个块")

for i in range(len(chunks)):
    print(f"第{i}个块的内容：")
    print(f"{chunks[i].page_content if len(chunks) > else} 没有第{i}个块")

SyntaxError: f-string: invalid syntax (2880443365.py, line 18)

In [10]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,        # 每个块的最大字符数（可根据文档调整）
    chunk_overlap=50,      # 块之间的重叠字符数，防止语义断裂
    length_function=len,   # 计算长度的函数，默认len
    separators=["\n\n", "\n", "。", "！", "？", "，", " ", ""]  # 优先按段落、句子分割
)

# 对文档进行切分
chunks = text_splitter.split_documents(documents)

# ========== 第三步：打印结果（修复语法错误） ==========
print(f"原始文档被切分成 {len(chunks)} 个块")

# 修复循环中的三元表达式，同时简化遍历方式
for i, chunk in enumerate(chunks):
    print(f"\n第{i}个块的内容：")
    # 正确的三元表达式：内容 if 存在 else 提示语
    print(chunk.page_content if chunk.page_content else "该块为空内容")

原始文档被切分成 10 个块

第0个块的内容：
浙江大学教务处通知汇编
═══════════════════════════════════════════════════════

【通知一】关于2025-2026学年秋冬学期本科课程第四轮选课的通知
发布时间：2025年10月30日
内容：2025-2026学年秋冬学期本科课程第四轮选课安排

第1个块的内容：
【通知二】2025级本科生主修专业确认工作的通知
发布时间：2025年10月30日
内容：2025级本科生主修专业确认工作启动

【通知三】关于本科生2024-2025学年春夏学期报到注册的温馨提示
发布时间：2025年1月13日
来源：外国语学院本科与继续教育科

主要内容：
一、报到注册时间
2024-2025学年春夏学期报到注册安排

第2个块的内容：
主要内容：
一、报到注册时间
2024-2025学年春夏学期报到注册安排

二、特殊情况处理
如有特殊情况不能按时返校的学生，须根据《浙江大学本科学生学籍管理办法》（浙大发本【2020】52号）第十二条的规定，向辅导员办理请假手续，申请暂缓注册，返校后及时将学生证交到东五307报到注册。

第3个块的内容：
三、重要提醒
如超过学校规定期限未注册超过2周，且未履行暂缓注册手续的，根据《浙江大学本科学生学籍管理办法》（浙大发本【2020】52号）第四十六条的规定，应给予退学处理。同时对经过核实的未注册且未办理请假手续的学生在2月28日前邮寄拟退学处理通知书。

第4个块的内容：
四、对外交流学生注册
2024-2025学年春夏学期参加线下对外交流的学生请及时和辅导员报备，按对外交流办理注册。派出前教学管理服务平台（http://zdbk.zju.edu.cn）需完成派出办结。在派出前务必根据学校相关要求，及时在教务网完成课程退选。交流期间不得同时修读浙大的课程。

联系方式：杨老师 yangfengzj@zju.edu.cn  0571-88981916

第5个块的内容：
联系方式：杨老师 yangfengzj@zju.edu.cn  0571-88981916

【通知四】关于教师审核学生课程免听申请的通知
发布时间：2024年2月23日

操作步骤：
登录http://zdbk.zju.edu.cn，在【选课管理】-【免听申请处理】栏目
选择